<a href="https://colab.research.google.com/github/kevin-blasiak-curtin/ISYS2001-Archive/blob/main/Module%2007%20-%20Directing%20Pandas/lab_ticket_pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab ticket: sales analysis with pandas

In the worksheet we used pandas to total transactions by category, and we checked its
answers against a loop we understood. Now you do the same kind of work on your own,
with a different dataset and no worked solution to lean on.

This ticket is pass/fail. It is not marked on how polished your code is. It is marked
on whether you can explain what your code does and why you did it that way. Expect to
be asked, out loud, about any line of it.

## The situation

You are looking at a month of sales for an office-supply company. Each row is one
sale. The file `sales_data.csv` has these columns:

- `Date` the day of the sale
- `Product` what was sold
- `Quantity` how many units were sold
- `Unit_Price` the price of one unit
- `Total_Sale` the value of that sale
- `Sales_Rep` who made the sale
- `Region` where it was sold

Your manager wants two things from you: which regions are bringing in the most money,
and how many units of each product are moving. Simple questions. The data will not be
quite as tidy as the questions.

## Before you write anything: look at the data

Open the file and read it as plain text first, the way we did in the worksheet. Do
not load it into pandas yet. Just look.

As you read, ask yourself which columns are numbers and which are text pretending to
be numbers. Some columns have dollar signs. Look carefully at *which* ones, and ask
whether a dollar sign makes sense for what that column is measuring.

In [14]:
# Read sales_data.csv as plain text and look at it.
# What do you notice about the columns? Which ones will need cleaning? Looking at the file as plain text shows that the Unit_Price, Total_Sale, and Quantity columns contain non-numeric symbols like $ and commas. Because of these formatting symbols, Pandas will read them as text strings (object) instead of numbers.

In [8]:
# Read sales_data.csv as plain text and look at it.
with open('sales_data.csv') as f:
    for line in f:
        print(line.strip())

Date,Product,Quantity,Unit_Price,Total_Sale,Sales_Rep,Region
2024-08-01,Business Analytics Software,$2,$2499.00,$4998.00,Sarah Chen,NSW
2024-08-01,Office Chair - Ergonomic,$15,$299.50,$4492.50,Michael Wong,VIC
2024-08-02,Standing Desk,$8,$599.00,$4792.00,Emma Thompson,QLD
2024-08-02,Laptop - Business Grade,$12,$1899.00,$22788.00,David Kumar,NSW
2024-08-03,Conference Table,$3,$1250.00,$3750.00,Lisa Park,WA
2024-08-03,Projector - 4K Business,$6,$799.95,$4799.70,Sarah Chen,NSW
2024-08-04,Whiteboard - Interactive,$4,$1899.00,$7596.00,Michael Wong,VIC
2024-08-05,Office Phone System,$10,$189.50,$1895.00,Emma Thompson,QLD
2024-08-05,Printer - Multifunction,$8,$449.00,$3592.00,David Kumar,NSW
2024-08-06,Security Camera System,$2,$1299.00,$2598.00,Lisa Park,WA
2024-08-07,Router - Business Grade,$25,$199.95,$4998.75,Sarah Chen,NSW
2024-08-08,Monitor - 27 inch,$18,$389.00,$7002.00,Michael Wong,VIC
2024-08-09,Desk Lamp - LED,$45,$79.95,$3597.75,Emma Thompson,QLD
2024-08-10,Filing Cabinet,$12,$249.

In [9]:
import csv

region_totals = {}

with open('sales_data.csv') as f:
    reader = csv.DictReader(f)
    for row in reader:
        # Strip the dollar sign and commas from Total_Sale before converting to float
        sale_text = row['Total_Sale'].replace('$', '').replace(',', '')
        sale = float(sale_text)

        region = row['Region']
        if region in region_totals:
            region_totals[region] += sale
        else:
            region_totals[region] = sale

# Sort the regions from highest total to lowest
ordered = sorted(region_totals.items(), key=lambda pair: pair[1], reverse=True)

for region, total in ordered:
    print(f"{region:<15} {total:>12.2f}")

NSW                 50902.75
VIC                 21085.50
QLD                 11904.30
WA                  10049.25


Write one or two sentences here, in plain words, about what you noticed. In
particular: is every column with a dollar sign actually an amount of money? What
would go wrong if you treated all of them the same way?

Not every column with a dollar sign represents money, like Quantity also has dollar signs due to a formatting error. If you treated them all the same and converted Quantity to a decimal (float) like money, you would get the wrong data type for counting whole items and could end up with strange fractional unit counts.

## Task 1: total sales by region, highest first

Load the file into a DataFrame, clean whatever needs cleaning so the sale values are
real numbers, and produce the total `Total_Sale` for each `Region`, ordered from
highest to lowest.

Plan it in plain words in the cell below before you write the pandas. If you use an
assistant to help with a line, give it your plan and make sure you can explain what
it hands back.

My plan, in plain words:
1. Load the sales_data.csv file into a pandas DataFrame.
2. Clean the Total_Sale column by removing the dollar signs ($) and commas (,), then convert the values into numeric floats.
3. Group the dataset by the Region column and calculate the total sum of Total_Sale for each region.
4. Sort the totals from highest to lowest and display the result.

In [30]:
# Task 1: total sales by region, highest first.
import pandas as pd

df = pd.read_csv('sales_data.csv')

df['Total_Sale'] = df['Total_Sale'].str.replace('$', '', regex=False)
df['Total_Sale'] = df['Total_Sale'].str.replace(',', '', regex=False)
df['Total_Sale'] = pd.to_numeric(df['Total_Sale'])

sales_by_region = df.groupby('Region')['Total_Sale'].sum().sort_values(ascending=False)

sales_by_region

,Total_Sale
Region,
NSW,50902.75
VIC,21085.50
QLD,11904.30
WA,10049.25


## Task 2: units sold per product

Now find how many units of each product were sold in total, using the `Quantity`
column.

This is where looking at the data first pays off. If your numbers come out looking
like money rather than counts of units, stop and work out why before you go on. A
correct answer here should read as sensible unit counts, not dollar amounts.

My plan, in plain words:
1. Clean the Quantity column by stripping dollar signs ($) and commas (,), then convert it into numeric values using pd.to_numeric().
2. Group the rows by Product, sum the Quantity for each product, and sort the totals from highest to lowest.

In [31]:
# Task 2: total units sold per product.
df['Quantity'] = df['Quantity'].str.replace('$', '', regex=False)
df['Quantity'] = df['Quantity'].str.replace(',', '', regex=False)
df['Quantity'] = pd.to_numeric(df['Quantity'])

product_totals = df.groupby('Product')['Quantity'].sum()
product_totals = product_totals.sort_values(ascending=False)

product_totals

,Quantity
Product,
Desk Lamp - LED,45
Keyboard - Wireless,35
Mouse - Ergonomic,35
Router - Business Grade,25
Monitor - 27 inch,18
Office Chair - Ergonomic,15
Laptop - Business Grade,12
Filing Cabinet,12
Office Phone System,10


## Task 3: one question of your own

Pick one more question you could answer from this data and answer it. It does not have
to be complicated. A few honest options:

- Which sales rep brought in the most revenue?
- What is the average sale value in each region?
- Which single product line is worth the most in total sales?

State your question, plan it in words, then write the pandas for it.

My question, and my plan:

Question: What is the average sale value in each region?

My plan:
1. Ensure the Total_Sale column is cleaned of dollar signs ($) and commas (,), then converted to numeric values.
2. Group the rows by Region and calculate the average (mean) Total_Sale for each region.
3. Sort the average values from highest to lowest and display the result.

In [35]:
# Task 3: your own question.
avg_sale_by_region = df.groupby('Region')['Total_Sale'].mean()
avg_sale_by_region = avg_sale_by_region.sort_values(ascending=False)

avg_sale_by_region

,Total_Sale
Region,
NSW,6362.84375
VIC,5271.37500
QLD,2976.07500
WA,2512.31250


## Reflection

Answer these honestly in a sentence or two each. This is the part that is actually
marked.

1. What did looking at the data first tell you that you would have missed if you had
   loaded it straight into pandas?

   Checking the raw data first helped me spot the dollar signs ($) and commas (,). If I had just loaded it directly into pandas, those columns would have been imported as text, which would've stopped me from doing any math on them.

2. Was there a point where pandas gave you an answer that looked wrong? How did you
   notice, and what did you do?

   Yeah, I ran into an AttributeError in Task 3. I realized it happened because the Total_Sale column was already turned into a number back in Task 1, but I was still trying to run string cleaning functions on it. I fixed it by dropping the extra cleaning steps and going straight to .mean().

3. Where in this ticket did you use an assistant, and how did you check that what it
   gave you was right?

   I used an assistant to troubleshoot the error and to understand why I needed two .str.replace() steps for the formatting. To make sure the advice was right, I checked the final data types and made sure the code matched the style from our course materials.


## Before you submit

- [ ] I looked at the raw data before loading it, and I noted what I saw.
- [ ] Task 1 produces total sales by region, highest first.
- [ ] Task 2 produces sensible unit counts per product, not dollar amounts.
- [ ] Task 3 states a question, a plan, and working code.
- [ ] I can explain every line I have written, including any an assistant helped with.
- [ ] My reflection answers are honest, including anything that went wrong.